Logisitic Regression Attempt 2
---
<b>Summary</b>  
This notebook uses the same workflow of the first LogReg attempt but does not use Regularization due to minimal affects from it before.

This notebook starts by creating a baseline logistic regression using a train, validate, and test set. The validation set was used to pick the threshold of 0.11 and compare it to the baseline of 0.5. The 0.11 threshold decreases False Negatives, but increases False Positives. 

Things to Consider:
- is it more important to detect as many possible defaults than avoiding the false alarms
- is missing a default more costly than if we incorrectly classified is as a default
- how should the model be used, to automatically reject loans or flag for mroe inspection/review

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score, precision_score, f1_score, recall_score

from sklearn.base import clone
from sklearn.metrics import roc_curve, roc_auc_score

from sklearn.calibration import calibration_curve


px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Dark24

In [2]:
# load in data
# df = pd.read_csv('../Loan_default.csv')
df = pd.read_csv('../Somaya/Loan_default_cleaned.csv')
df

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DebtToIncomeRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,LoanToIncomeRatio
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,0.588262
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,2.467481
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,1.534154
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,1.412638
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,0.447179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0,5.547329
255343,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1,3.655208
255344,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0,2.455718
255345,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0,0.711734


In [3]:
df.columns # target column = "default"

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DebtToIncomeRatio',
       'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
       'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default',
       'LoanToIncomeRatio'],
      dtype='str')

In [4]:
df.dtypes

Age                    int64
Income                 int64
LoanAmount             int64
CreditScore            int64
MonthsEmployed         int64
NumCreditLines         int64
InterestRate         float64
LoanTerm               int64
DebtToIncomeRatio    float64
Education                str
EmploymentType           str
MaritalStatus            str
HasMortgage              str
HasDependents            str
LoanPurpose              str
HasCoSigner              str
Default                int64
LoanToIncomeRatio    float64
dtype: object

Building the Logisitc Regression

In [5]:
# save the target column in y
y = df['Default']

# save the predictors or features in x
x = df[['Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DebtToIncomeRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'LoanToIncomeRatio']]

# save the categorical variables for preprocessing
cats = ['Education', 'EmploymentType','MaritalStatus',
        'HasMortgage', 'HasDependents','LoanPurpose','HasCoSigner']

# save the numerical variables for preprocessing
nums = ['Age', 'Income', 'LoanAmount',   'CreditScore',
        'MonthsEmployed','NumCreditLines','InterestRate', 'LoanTerm',
        'DebtToIncomeRatio', 'LoanToIncomeRatio']

Splitting into train, test, and validation. Will use validation set to find the threshold

In [6]:
# split data into training and test
x_train_val, x_test, y_train_val, y_test = train_test_split(
    x, y, test_size=0.2, random_state=6021, stratify=y
)

# split data into training and test
x_train, x_val, y_train, y_val = train_test_split(
    x_train_val, y_train_val, test_size=0.25, random_state=6021, stratify=y_train_val
)

display(pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Rows': [len(y_train), len(y_val), len(y_test)],
    'Default rate' : [y_train.mean(), y_val.mean(), y_test.mean()]
}))

,Split,Rows,Default rate
0,Train,153207,0.116124
1,Validation,51070,0.116135
2,Test,51070,0.116135


In [7]:
# create the order of the pipeline
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop='first', handle_unknown='ignore'), cats),
    ("numeric", StandardScaler(), nums)
])

In [8]:
# create a log reg model
logreg=LogisticRegression(max_iter=5000, random_state=123)

# create a preporcessing pipeline
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])

# fit
pipe.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['Age','Income','LoanAmount',...,'LoanPurpose','HasCoSigner', 'LoanToIncomeRatio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pas

In [9]:
# get the predicted probabilities
p_train = pipe.predict_proba(x_train)[:,1]
p_val = pipe.predict_proba(x_val)[:,1]
p_test = pipe.predict_proba(x_test)[:,1]

# get the log odds
log_odds_train = pipe.decision_function(x_train)
log_odds_val = pipe.decision_function(x_val)
log_odds_test = pipe.decision_function(x_test)


In [10]:
feature_names = pipe.named_steps['preprocess'].get_feature_names_out()
beta = pipe.named_steps['model'].coef_[0]
coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': beta,
    'Odds Ratio' : np.exp(beta)
})
print('Intercept: ', pipe.named_steps['model'].intercept_[0])
coefs.sort_values('Coefficient', ascending=False)


Intercept:  -2.091008632947802


,Feature,Coefficient,Odds Ratio
21,numeric__InterestRate,0.461079,1.585784
24,numeric__LoanToIncomeRatio,0.460122,1.584267
5,encoder__EmploymentType_Unemployed,0.440038,1.552767
3,encoder__EmploymentType_Part-time,0.282399,1.326308
4,encoder__EmploymentType_Self-employed,0.242719,1.274710
20,numeric__NumCreditLines,0.104132,1.109747
0,encoder__Education_High School,0.080002,1.083289
23,numeric__DebtToIncomeRatio,0.054313,1.055815
10,encoder__LoanPurpose_Business,0.042884,1.043817
16,numeric__Income,0.012276,1.012352


In [11]:
# defining a function that will get the pred, the TN, ..., and return all the metrics 
def metrics_at(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {'Threshold': threshold, 'TP':tp, 'FP':fp, 'FN':fn, 'TN':tn,
            'Accuracy':accuracy_score(y_true,pred),
            'Precision':precision_score(y_true,pred,zero_division=0),
            'Recall':recall_score(y_true,pred,zero_division=0),
            'F1':f1_score(y_true,pred,zero_division=0),
            'ROC AUC':roc_auc_score(y_true,probabilities),
            'Log loss':log_loss(y_true,probabilities)}

# metrics at threshold = 0.5 for the training data and the validation data
baseline = pd.DataFrame([
    {'Split':'Train', **metrics_at(y_train,p_train,0.50)},
    {'Split':'Validation', **metrics_at(y_val,p_val,0.50)}])
display(baseline.round(4))

,Split,Threshold,TP,FP,FN,TN,Accuracy,Precision,Recall,F1,ROC AUC,Log loss
0,Train,0.5,1187,766,16604,134650,0.8866,0.6078,0.0667,0.1202,0.7557,0.3120
1,Validation,0.5,390,226,5541,44913,0.8871,0.6331,0.0658,0.1191,0.7608,0.3098


A high accuracy and low recall because there are less actual default cases than non-default cases

In [12]:
# testing a range of different thresholds on the validation data 
thresholds = np.round(np.arange(0.05,0.951,0.01),2)
validation_results = pd.DataFrame([
    metrics_at(y_val,p_val,t) for t in thresholds
])
# show the metrics for the thresholds that satisfy the threholds in this list 
display(validation_results.loc[validation_results['Threshold'].isin([0.10,0.15,0.20,0.25,0.30,0.50,0.70,0.90])].round(3))
# plot the mertics 
fig = go.Figure()
for col,label in [('FP','False positives (false alarms)'),('FN','False negatives (missed defaults)')]:
    fig.add_trace(go.Scatter(x=validation_results['Threshold'],
                             y=validation_results[col],mode='lines',name=label))
fig.update_layout(title='Validation errors vs classification threshold',
                  xaxis_title='Threshold',yaxis_title='Number of borrowers')
fig.show()

,Threshold,TP,FP,FN,TN,Accuracy,Precision,Recall,F1,ROC AUC,Log loss
5,0.10,4434,16771,1497,28368,0.642,0.209,0.748,0.327,0.761,0.31
10,0.15,3441,9506,2490,35633,0.765,0.266,0.580,0.365,0.761,0.31
15,0.20,2594,5446,3337,39693,0.828,0.323,0.437,0.371,0.761,0.31
20,0.25,1897,3202,4034,41937,0.858,0.372,0.320,0.344,0.761,0.31
25,0.30,1381,1869,4550,43270,0.874,0.425,0.233,0.301,0.761,0.31
45,0.50,390,226,5541,44913,0.887,0.633,0.066,0.119,0.761,0.31
65,0.70,76,23,5855,45116,0.885,0.768,0.013,0.025,0.761,0.31
85,0.90,1,0,5930,45139,0.884,1.000,0.000,0.000,0.761,0.31


In [13]:
MIN_RECALL = 0.70  # arbitrary choice, saying we want a min recall of .7 based off of outside cost, ex. FN are expensive
eligible = validation_results[validation_results['Recall'] >= MIN_RECALL]
if eligible.empty:
    print('No threshold meets the recall target; choosing highest validation F1 instead.')
    eligible = validation_results
chosen = eligible.sort_values(['F1','Precision'],ascending=False).iloc[0]
selected_threshold = float(chosen['Threshold'])
print('Selected threshold:',selected_threshold)
display(chosen.to_frame('Validation result').T.round(4))

Selected threshold: 0.11


,Threshold,TP,FP,FN,TN,Accuracy,Precision,Recall,F1,ROC AUC,Log loss
Validation result,0.11,4236.0,14909.0,1695.0,30230.0,0.6749,0.2213,0.7142,0.3379,0.7608,0.3098


Threshold of 0.11 meets 0.70 min recall, minimizes out false negatives

In [14]:
# comparing default vs our select threshold and their metrics
final_metrics = pd.DataFrame([
    {'Rule':'Default 0.50', **metrics_at(y_test,p_test,0.50)},
    {'Rule':'Validation-selected', **metrics_at(y_test,p_test,selected_threshold)}
])
display(final_metrics.round(4))
for rule,t in [('Default 0.50',0.50),('Validation-selected',selected_threshold)]:
    pred=(p_test>=t).astype(int)
    cm=confusion_matrix(y_test,pred,labels=[1,0])
    cm_df=pd.DataFrame(cm,index=['Actual default','Actual no default'],
                       columns=['Predicted default','Predicted no default'])
    fig=px.imshow(cm_df,text_auto=True,color_continuous_scale='Blues',
                  title=f'Test confusion matrix — {rule} (t={t:.2f})')
    fig.show()

,Rule,Threshold,TP,FP,FN,TN,Accuracy,Precision,Recall,F1,ROC AUC,Log loss
0,Default 0.50,0.50,404,258,5527,44881,0.8867,0.6103,0.0681,0.1226,0.7582,0.3107
1,Validation-selected,0.11,4187,14934,1744,30205,0.6734,0.2190,0.7060,0.3343,0.7582,0.3107


In [15]:
fpr,tpr,_=roc_curve(y_test,p_test)
fig=go.Figure()
fig.add_trace(go.Scatter(x=fpr,y=tpr,mode='lines',name=f'Logit (AUC={roc_auc_score(y_test,p_test):.3f})'))
fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',line=dict(dash='dash'),name='Random baseline'))
fig.update_layout(title='Test ROC curve',xaxis_title='False positive rate',yaxis_title='True positive rate')
fig.show()

prob_true,prob_pred=calibration_curve(y_test,p_test,n_bins=10,strategy='quantile')
fig=go.Figure()
fig.add_trace(go.Scatter(x=prob_pred,y=prob_true,mode='lines+markers',name='Logistic regression'))
fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',line=dict(dash='dash'),name='Perfect calibration'))
fig.update_layout(title='Test calibration',xaxis_title='Mean predicted default probability',
                  yaxis_title='Observed default frequency')
fig.show()